In [ ]:
# =====================================================================
# Dynamic-Room Training Data Generator (Verified Engine)
#   Uses exact 4D engine math per room
#   Grid scales to room size (~0.05m spacing)
#   200 rooms × ~150 windows = 30,000 samples
# =====================================================================

import torch, numpy as np, math, time
from scipy.stats import gaussian_kde
from scipy.stats.qmc import LatinHypercube
from tqdm import tqdm
from IPython.display import display, HTML, clear_output
def status(msg): display(HTML(f'<b style="color:#16213e">{msg}</b>')); return time.time()
import multiprocessing, concurrent.futures

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ==========================================
# Config
# ==========================================
N_ROOMS, N_SAMPLES = 200, 30000
ROOM_MIN, ROOM_MAX = 2.0, 20.0
ROOM_Z = 3.0

# ==========================================
# Fixed Parameters (identical to 4D engine)
# ==========================================
x_BS,y_BS,z_BS=10.0,-100.0,-10.0
E=8;d_B=0.075;lambda_val=0.075;L1=2;d_ref=abs(y_BS)*1.5
P_BS_dBm=40.0;R_th=0.2;N0_dBm_Hz=-174.0;B=20e6;p_m=1./5.;n_users=5
P_BS=10**(P_BS_dBm/10.)*1e-3;N0=10**(N0_dBm_Hz/10.)*1e-3*B
Z_HEIGHTS=[1.5,1.625,1.75,1.875,2.0];N_Z=len(Z_HEIGHTS)

# ==========================================
# RWP + KDE Worker (pure NumPy, multiprocessing safe)
# ==========================================
def gen_rwp_traj(rx,ry,sim_time,dt=10,nu=5):
    rng=np.random;rng.seed(777)
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot' else tau_n
    def sp(r):return v_h if r=='hot' else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

def cpu_worker(rxv,ryv):
    rx,ry=float(rxv),float(ryv)
    nx,ny=max(40,int(rx/0.05)),max(40,int(ry/0.05))
    xv=np.linspace(0,rx,nx);yv=np.linspace(0,ry,ny)
    Xg,Yg=np.meshgrid(xv,yv,indexing='ij')
    xyf=np.stack([Xg.flatten(),Yg.flatten()],axis=1)
    gl_np=[]
    for zh in Z_HEIGHTS: gl_np.append(np.stack([Xg.flatten(),Yg.flatten(),np.full_like(Xg.flatten(),zh)],axis=1))
    gl_np=np.concatenate(gl_np,axis=0);Ng=gl_np.shape[0]
    et=gen_rwp_traj(rx,ry,100000,10)
    kde=gaussian_kde(et[:,:2].T,bw_method='scott')
    wxy=kde(xyf.T).flatten();wxy=wxy/wxy.sum()
    w_full=np.tile(wxy,N_Z);gw_np=w_full/w_full.sum()
    np.random.seed(42)
    ps_np=2*np.pi*np.random.rand(Ng);tt_np=-np.pi+2*np.pi*np.random.rand(Ng)
    eta_np=10**((-15+5*np.random.rand(Ng))/10)
    return {'gl':gl_np,'gw':gw_np,'ps':ps_np,'tt':tt_np,'eta':eta_np}

# ==========================================
# Pre-build rooms (multiprocessing)
# ==========================================
print(f'\nPre-sampling {N_ROOMS} room dimensions...')
sampler_room=LatinHypercube(d=2,seed=999)
U_rooms=sampler_room.random(n=N_ROOMS)
room_pairs=np.column_stack([ROOM_MIN+U_rooms[:,0]*(ROOM_MAX-ROOM_MIN),
                             ROOM_MIN+U_rooms[:,1]*(ROOM_MAX-ROOM_MIN)])
room_pairs=np.round(room_pairs,2)

nw=min(multiprocessing.cpu_count(),16)
print(f'  Using {nw} CPU workers')
t0=time.time();raw={}
with concurrent.futures.ProcessPoolExecutor(max_workers=nw) as ex:
    futs={ex.submit(cpu_worker,rp[0],rp[1]):(rp[0],rp[1]) for rp in room_pairs}
    for f in tqdm(concurrent.futures.as_completed(futs),total=N_ROOMS,desc='KDE'):
        raw[futs[f]]=f.result()
print(f'  CPU done in {time.time()-t0:.0f}s. Uploading to GPU...')

na_np=np.arange(E,dtype=np.float32);dyB_np=np.float32(0.0-y_BS)
v1c_np=lambda_val/(4*math.pi);v1pc_np=-(2*math.pi/lambda_val);oE_np=1/math.sqrt(E)
_CACHE={}
for key,data in raw.items():
    _CACHE[key]=(torch.tensor(data['gl'],dtype=torch.float32,device=device),
        torch.tensor(data['gw'],dtype=torch.float32,device=device),
        torch.tensor(data['ps'],dtype=torch.float32,device=device),
        torch.tensor(data['tt'],dtype=torch.float32,device=device),
        torch.tensor(data['eta'],dtype=torch.float32,device=device),
        torch.tensor(na_np,dtype=torch.float32,device=device),
        torch.tensor(dyB_np,dtype=torch.float32,device=device),v1c_np,v1pc_np,oE_np)
print(f'  GPU cache ready. Rooms: {len(_CACHE)}')

# ==========================================
# Verified channel + outage (per-window, identical to 4D engine)
# ==========================================
@torch.no_grad()
def outage_verified(X):
    """X: (N,6) [xc,zc,Lx,Lz,rx,ry]"""
    out=[]
    for i in range(0,len(X),200):
        b=torch.tensor(X[i:i+200],dtype=torch.float32,device=device)
        bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            key=(round(b[j,4].item(),2),round(b[j,5].item(),2))
            gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=_CACHE[key]
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3]
            xg,yg,zg=gl[:,0],gl[:,1],gl[:,2];Ng=xg.shape[0]
            # Channel (exact 4D mirror)
            dx=xc-x_BS;dy=dyB;dz=zc-z_BS
            Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            x1,z1=xc-Lx/2,zc-Lz/2;x2,z2=xc-Lx/2,zc+Lz/2;x3,z3=xc+Lx/2,zc-Lz/2;x4,z4=xc+Lx/2,zc+Lz/2
            def rd(xs,zs):dd=xs-x_BS;dz_=zs-z_BS;L=torch.sqrt(dd**2+dy**2+dz_**2);return dd/L,dy/L,dz_/L
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            dux=xg-x_BS;duy=yg-y_BS;duz=zg-z_BS;Lu=torch.sqrt(dux**2+duy**2+duz**2)
            uux=dux/Lu;uuz=duz/Lu
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
            d2x=xg-xc;d2y=yg;d2z=zg-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
            ax=(1-il)*(kx-t1);az=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lambda_val)*Lx*ax);sz=torch.sinc((math.pi/lambda_val)*Lz*az)
            p1=(2*math.pi/lambda_val)*d_B*na*torch.sin(th)
            a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m=v1c/Rw;v1p=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p))
            H1=v1*a1.conj()
            p2=(2*math.pi/lambda_val)*d_B*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
            a2=oE*torch.complex(torch.cos(p2),torch.sin(p2))
            v2m=eta*(lambda_val/(4*math.pi*d_ref));v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps))
            H2=v2.unsqueeze(1)*a2.conj()
            Ht=math.sqrt(E/L1)*(H1.unsqueeze(0)+H2)
            fm=(Lx*Lz*sx*sz)/(lambda_val*Ru)
            fp=torch.tensor(-(2*math.pi/lambda_val),device=device)*(kx*xc+kz*zc)-(math.pi/2)
            fc=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
            He=Ht*fc.unsqueeze(1);Hw=torch.sum(He,dim=1)/math.sqrt(E)
            dp=(torch.abs(Hw)**2)*p_m*P_BS;it=(n_users-1)*dp;sr=dp/(it+N0)
            ab=math.pi/3.0;Ses=torch.zeros(Ng,device=device)
            rn=torch.sqrt((xg-xc)**2+yg**2+(zg-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xg-xc)*math.cos(a)+yg*math.sin(a));cp=dot/rn
                ph_b=torch.acos(torch.clamp(cp,0,1));Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi
                Ses=Ses+torch.clamp(Se,1./3.,1.0)
            sr_se=((Ses/4)*dp)/((Ses/4)*it+N0)
            bits=(torch.log2(1+sr_se)<R_th).float();bo[j]=torch.sum(bits*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

print('\nPhysics engine ready.')

# ==========================================
# Generate window samples
# ==========================================
print(f'Generating {N_SAMPLES} window samples...')
np.random.seed(123)
room_idx=np.random.randint(0,N_ROOMS,N_SAMPLES)
rx_s=room_pairs[room_idx,0];ry_s=room_pairs[room_idx,1]
U_w=np.random.rand(N_SAMPLES,4)
Lx_s=0.05+U_w[:,0]*(rx_s-0.05);Lz_s=0.05+U_w[:,1]*2.95
xc_s=Lx_s/2+U_w[:,2]*(rx_s-Lx_s);zc_s=Lz_s/2+U_w[:,3]*(3.0-Lz_s)
Xa=np.column_stack([xc_s,zc_s,Lx_s,Lz_s,rx_s,ry_s])

t0=time.time();Ya=[]
for k in tqdm(range(0,N_SAMPLES,500)): Ya.append(outage_verified(Xa[k:k+500]))
Ya=np.concatenate(Ya)
print(f'Done in {time.time()-t0:.0f}s')
print(f'Feasible (<10%): {(Ya<0.10).sum()} ({(Ya<0.10).sum()/len(Ya)*100:.1f}%)')

# ==========================================
# Feature engineering (17 features)
# ==========================================
xc=Xa[:,0];zc=Xa[:,1];Lx=Xa[:,2];Lz=Xa[:,3];rx=Xa[:,4];ry=Xa[:,5]
xc_rel=xc/rx;Lx_rel=Lx/rx;area_ratio=(Lx*Lz)/(rx*3)
dx=xc-10.0;dy_r=ry+100.0;dz_=zc+10.0;dist_bs=np.sqrt(dx**2+dy_r**2+dz_**2)
alpha_x=dx/dist_bs;alpha_z=dz_/dist_bs
mL=xc-Lx/2;mR=rx-(xc+Lx/2);mB=zc-Lz/2;mT=3-(zc+Lz/2);asp=Lx/(Lz+1e-6)
Xe=np.column_stack([xc,zc,Lx,Lz,rx,ry,xc_rel,Lx_rel,area_ratio,dist_bs,alpha_x,alpha_z,
                     mL,mR,mB,mT,asp])
cols=['xc','zc','Lx','Lz','rx','ry','xc_rel','Lx_rel','area_ratio','dist_bs_win','alpha_x','alpha_z',
      'margin_left','margin_right','margin_bottom','margin_top','aspect_ratio','outage']
np.savetxt('dynamic_train_enriched.csv',np.column_stack([Xe,Ya]),delimiter=',',header=','.join(cols),comments='')
print(f'Saved dynamic_train_enriched.csv ({Xe.shape[1]} features, {len(Ya)} rows)')
